## (0) Setup
Go through the setup instruction on our [Github Wiki](https://github.com/pro3d-space/PRo3D/wiki/Interactive-Co%E2%80%90Registration-Workflow). Jupyter setup concludes with executing the import cell 👇.

In [ ]:
# imports
from typing import Any
import requests
import json
import open3d as o3d
import numpy as np
import subprocess

from pro3d_api import Pro3DClient
import data_io as io
import o3d_uitls as o3du

If not already running, start your PRo3D Instance with the following command line parameters

```
PRo3D.Viewer.exe --remoteApi --port 4321 
```

In [ ]:
# create api client Pro3D instance
client = Pro3DClient(port=4321)

# apply trafo from protocol

In [ ]:
# Apply Trafo from Protocol to MOV Surface

## Fill in correct Trafo here !!!
trafo_str = """
9.96967123e-01 -1.98354618e-02  7.52536418e-02 -1.65627852e+04
2.16794204e-02  9.99482458e-01 -2.37659369e-02  1.18862640e+04
-7.47432865e-02  2.53253131e-02  9.96881171e-01 -2.33601240e+04
0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00
"""

## Convert to NumPy array
trafo = np.fromstring(trafo_str, sep=' ').reshape((4, 4))

👈 Select MOV surface

In [ ]:
## Apply
client.apply_transformation_to_selected_surface(trafo, verbose=True)

# prepare cutouts

👈 select cutout annotation

👈 Select REF surface

In [ ]:
# REF Surface Cutout

json_data = client.query_annotation_as_json()
points = np.array(json.loads(json_data))

## Create o3d point cloud object from cutout points
ref_pcd = o3du.create_point_cloud_from_array(points)

## offset transforation for large coordinates
offset = ref_pcd.get_center()

ref_pcd.translate(-offset)
o3d.io.write_point_cloud("ref.ply", ref_pcd)

👈 Select MOV surface

In [ ]:
json_data = client.query_annotation_as_json()
points = np.array(json.loads(json_data))

# Create o3d point cloud object from cutout points
mov_pcd = o3du.create_point_cloud_from_array(points)

👈 Select REF_AI surface

In [ ]:
json_data = client.query_annotation_as_json()
points = np.array(json.loads(json_data))

# Create o3d point cloud object from cutout points
ref_ai_pcd = o3du.create_point_cloud_from_array(points)

In [ ]:
mov_pcd.transform(T)

offset = mov_pcd.get_center()  # or any reference origin

mov_pcd.translate(-offset)
ref_pcd.translate(-offset)

o3d.io.write_point_cloud("mov.ply", mov_pcd)
o3d.io.write_point_cloud("ref.ply", ref_pcd)

In [ ]:
o3du.run_m3c2(
    "C:\\Program Files\\CloudCompare\\CloudCompare.exe",
    "ref.ply",
    "mov.ply",
    "param_file.txt")